In [ ]:
#| default_exp compact

# llmsurgery.compact

In [ ]:
#| export
from fastcore.utils import *
from functools import cache
from fastllm.chat import mk_msg, mk_msgs
from fastllm.types import PartType,Part,Msg

import re,tiktoken

In [ ]:
from fastcore.test import *

llmsurgery.compact defines a concise format for compacted LLM conversations. This is important because long conversations need to preserve the theory, decisions, and evidence needed for useful continuation while using far fewer tokens. We need it to support readable canonical rendering, bounded token budgets, tool calls and results, truncation, and repeated compaction.

## Compact DSL

We'll start with the compact DSL, which takes fastllm's canonical conversation data structures and renders them into this concise format.

First we will define and inspect the sample conversation using fastllm’s canonical Msg and Part data structures as our basis.

In [ ]:
#| export
def tool_part(tool_name, **arguments):
    "Create a canonical tool-use part"
    return Part(PartType.tool_use, data={'name':tool_name, 'arguments':arguments})

def tool_result(text):
    "Create a canonical tool-result message"
    return Msg('tool', [Part(PartType.tool_result, text=text)])

In [ ]:
python_str = 'files = ["a.py", "b.py"]'
bash_str = 'printf "ready\\n"\nls -la'
generic_args = {'path':'fixture.json', 'lines':[1, 2]}
res_str = 'first line\n\nsecond line\nfinal line'

python_part = tool_part('mcp__clikernel__execute', code=python_str)
bash_part = tool_part('Bash', command=bash_str)
generic_part = tool_part('inspect', **generic_args)

In [ ]:
turn1 = [
    mk_msg('List the project files, use the project skill, then run a shell check.'),
    mk_msg(['I will inspect the project.', python_part], role='assistant'),
    tool_result('files = ["a.py", "b.py"]'),
    mk_msg([tool_part('Skill', skill='project-files')], role='assistant'),
    tool_result('Skill loaded: project-files'),
    mk_msg('Base directory for this skill: /skills/project-files\n\nInspect project files before editing.'),
    mk_msg([bash_part], role='assistant'),
    tool_result('ready'),
]

In [ ]:
turn2 = [
    mk_msg('Keep the agreed names; do not rename the fixture.'),
    mk_msg(['Understood. I will verify the final state.', generic_part], role='assistant'),
    tool_result(res_str),
    mk_msg('The fixture is short, readable, and ready for the renderer.', role='assistant'),
]

In [ ]:
conversation = turn1 + turn2
len(conversation)

12

In [ ]:
turn1

[Msg(role='user', content=[Part(type=<PartType.text: 'text'>, text='List the project files, use the project skill, then run a shell check.', data=None)]),
 Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='I will inspect the project.', data=None), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'name': 'mcp__clikernel__execute', 'arguments': {'code': 'files = ["a.py", "b.py"]'}})]),
 Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='files = ["a.py", "b.py"]', data=None)]),
 Msg(role='assistant', content=[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'name': 'Skill', 'arguments': {'name': 'project-files'}})]),
 Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='Skill loaded: project-files', data=None)]),
 Msg(role='user', content=[Part(type=<PartType.text: 'text'>, text='Base directory for this skill: /skills/project-files\n\nInspect project files before editing.', data=None)]),


In [ ]:
turn2

[Msg(role='user', content=[Part(type=<PartType.text: 'text'>, text='Keep the agreed names; do not rename the fixture.', data=None)]),
 Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Understood. I will verify the final state.', data=None), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'name': 'inspect', 'arguments': {'path': 'fixture.json', 'lines': [1, 2]}})]),
 Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='first line\n\nsecond line\nfinal line', data=None)]),
 Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='The fixture is short, readable, and ready for the renderer.', data=None)])]

We use a *token* budget (based on OpenAI's public tokenizer, which should be at least indicative of other LLM's too), and truncate the *middle*, since the start and end are often the interesting bits.

In [ ]:
#| export
_compact_enc_name = 'o200k_base'

@cache
def compact_enc(): return tiktoken.get_encoding(_compact_enc_name)

def enc_toks(s, enc=None):
    "Encode `s` into tokens."
    if enc is None: enc = compact_enc()
    return enc.encode(s, disallowed_special=())

def len_toks(s, enc=None):
    "Number of tokens in `s`."
    return len(enc_toks(s, enc))

In [ ]:
enc = compact_enc()

In [ ]:
#| export
def trunctoks_mid(s, max_toks, enc=None, mark=' … '):
    """Truncate the middle of `s` to at most `max_toks` tokens."""
    if enc is None: enc = compact_enc()
    if max_toks <= 0: return ''
    toks = enc_toks(s, enc)
    if len(toks) <= max_toks: return s
    mtoks = enc_toks(mark, enc)[:max_toks]
    n = max_toks - len(mtoks)
    a, b = (n + 1) // 2, n // 2
    tail = toks[-b:] if b else []
    return enc.decode(toks[:a] + mtoks + tail)

`compact_text` renders ordinary user or assistant prose with a marker and a token limit. It is the simplest DSL conversion: delimit the text, then use middle truncation so both the beginning and ending remain available when the budget is exceeded.

In [ ]:
#| export
def compact_text(s, mark, max_toks, enc=None):
    """Render delimited prose within `max_toks`."""
    return trunctoks_mid(f'{mark} {s.strip()} {mark}', max_toks, enc)

In [ ]:
text = compact_text('beginning ' + 'middle ' * 100 + ' ending', '§', 20)
test(text, '§ beginning', str.startswith)
test(text, 'ending §', str.endswith)
assert len_toks(text) <= 20, text
print(text)

§ beginning middle middle middle middle middle middle middle …  middle middle middle middle middle middle  ending §


`compact_result` renders tool output compactly. It replaces internal newlines with `¶`, prefixes the result with `>`, and applies the same middle-truncation policy so the output remains useful within a token budget. An empty result is represented by the `>` marker alone.

In [ ]:
#| export
def compact_result(s, max_toks, enc=None):
    "Render a tool result within `max_toks`."
    text = ' ¶ '.join(line.strip() for line in s.splitlines() if line.strip())
    return trunctoks_mid(f'> {text}' if text else '>', max_toks, enc)

In [ ]:
result = compact_result(res_str, 20)
empty = compact_result('', 20)
test_eq(result, '> first line ¶ second line ¶ final line')
test_eq(empty, '>')
print(result)
print(empty)

> first line ¶ second line ¶ final line


>


`fenced_code` renders Python tool-call code as Markdown without allowing backticks inside the code to close the fence. It chooses a fence longer than any run of backticks in the source, then applies the shared token budget.

In [ ]:
#| export
def fenced_code(code, max_toks, enc=None):
    "Render Python in a safe Markdown fence within `max_toks`."
    code = code.strip()
    runs = re.findall(r'^ {0,3}(`{3,})[ \t]*$', code, flags=re.MULTILINE)
    fence = '`' * max(3, max(map(len, runs), default=0) + 1)
    pre,suf = f'{fence}python\n',f'\n{fence}'
    full = pre + code + suf
    if len_toks(full, enc) <= max_toks: return full
    if len_toks(pre + suf, enc) > max_toks:
        raise ValueError('Token budget is too small for a fenced block')
    body_toks = min(len_toks(code, enc), max_toks)
    while body_toks:
        body = trunctoks_mid(code, body_toks, enc)
        result = pre + body + suf
        if len_toks(result, enc) <= max_toks: return result
        body_toks -= 1
    return pre + suf

In [ ]:
simple = fenced_code(python_str, 40)
with_inline_backticks = fenced_code("s = '```text```'", 40)
with_fence_line = fenced_code("before\n```\nafter", 40)
truncated = fenced_code(python_str * 20, 20)
test_eq(simple, f'```python\n{python_str}\n```')
test_eq(with_inline_backticks, "```python\ns = '```text```'\n```")
test_eq(with_fence_line, "````python\nbefore\n```\nafter\n````")
test(truncated, '```python\n', str.startswith)
test(truncated, '\n```', str.endswith)
assert len_toks(truncated) <= 20, truncated
print(simple)

```python
files = ["a.py", "b.py"]
```


In [ ]:
print(with_inline_backticks)

```python
s = '```text```'
```


`bash_call` renders a Bash tool call as a compact shell command. It uses `!` as the marker, collapses non-empty command lines with `¶`, and applies the shared token budget.

In [ ]:
#| export
def bash_call(command, max_toks, enc=None):
    "Render a Bash command within `max_toks`."
    text = '! ' + ' ¶ '.join(line.strip() for line in command.splitlines() if line.strip())
    return trunctoks_mid(text, max_toks, enc)

In [ ]:
bash = bash_call(bash_str, 30)
test_eq(bash, '! printf "ready\\n" ¶ ls -la')
print(bash)

! printf "ready\n" ¶ ls -la


`generic_call` renders non-Bash tool calls as `▶ name(args)`. It formats named arguments compactly, replaces embedded newlines with `¶`, and applies the shared token budget.

In [ ]:
#| export
def generic_call(name, args, max_toks, enc=None):
    "Render a generic tool call within `max_toks`."
    vals = ', '.join(f"{k}={v!r}".replace('\\n', ' ¶ ') for k,v in args.items())
    return trunctoks_mid(f'▶ {name}({vals})', max_toks, enc)

In [ ]:
call = generic_call(generic_part.data['name'], generic_part.data['arguments'], 40)
test_eq(call, "▶ inspect(path='fixture.json', lines=[1, 2])")
print(call)

▶ inspect(path='fixture.json', lines=[1, 2])


`compact_call` dispatches each tool-use part to its specialized renderer: clikernel calls use fenced Python, Bash uses the `!` form, and all other tools use `▶ name(args)`. This keeps tool-specific formatting out of the higher-level message renderer.

In [ ]:
#| export
def compact_call(p, max_toks, enc=None):
    "Render a tool-use part within `max_toks`."
    name, args = p.data['name'], p.data['arguments']
    if name == 'mcp__clikernel__execute': return fenced_code(args['code'], max_toks, enc)
    if name == 'Bash': return bash_call(args.get('command', ''), max_toks, enc)
    return generic_call(name, args, max_toks, enc)

In [ ]:
parts = [python_part, bash_part, generic_part]
rendered = [compact_call(p, 40) for p in parts]
test_eq(rendered, [simple, bash, call])
print('\n'.join(rendered))

```python
files = ["a.py", "b.py"]
```
! printf "ready\n" ¶ ls -la
▶ inspect(path='fixture.json', lines=[1, 2])


### Message rendering

User messages are rendered part by part with `§`. Injected skill contents are infrastructure rather than conversation-specific evidence, but experience shows a resumed model won't re-read them unprompted, so each is replaced by a marker carrying its own instruction - and the wrapping document ends by listing the exact `Skill(...)` calls to make (below).

In [ ]:
#| export
def compact_user_part(p, max_toks, enc=None):
    "Render one user-text part, replacing injected skill contents."
    text = '…skill text compacted: re-invoke before relying on it…' if p.text.startswith('Base directory for this skill:') else p.text
    return compact_text(text, '§', max_toks, enc)

def compact_user(m, max_toks, enc=None):
    "Render the text parts of a user message."
    return '\n'.join(compact_user_part(p, max_toks, enc)
        for p in m.content if p.type == PartType.text)

In [ ]:
user = compact_user(turn1[0], 100)
skill = compact_user(turn1[5], 100)
test_eq(user, '§ List the project files, use the project skill, then run a shell check. §')
test_eq(skill, '§ …skill text compacted: re-invoke before relying on it… §')
print(user)
print(skill)

§ List the project files, use the project skill, then run a shell check. §


§ …compacted… §


Assistant messages may interleave prose and tool calls. Text parts use `»`, while tool-use parts are delegated to `compact_call`; preserving their order keeps the response readable as a sequence.

In [ ]:
#| export
def compact_asst_part(p, text_toks, call_toks, enc=None):
    "Render one assistant part."
    if p.type == PartType.text: return compact_text(p.text, '»', text_toks, enc)
    if p.type == PartType.tool_use: return compact_call(p, call_toks, enc)

def compact_asst(m, text_toks, call_toks, enc=None):
    "Render an assistant message."
    parts = (compact_asst_part(p, text_toks, call_toks, enc) for p in m.content)
    return '\n'.join(filter(None, parts))

In [ ]:
assistant = compact_asst(turn1[1], 100, 40)
expected = f'» I will inspect the project. »\n{simple}'
test_eq(assistant, expected)
print(assistant)

» I will inspect the project. »
```python
files = ["a.py", "b.py"]
```


Tool messages contain tool-result parts. The top-level dispatcher selects the renderer from the canonical message role, giving the rest of the compaction pipeline one entry point for every message.

In [ ]:
#| export
def compact_tool(m, max_toks, enc=None):
    "Render the result parts of a tool message."
    return '\n'.join(compact_result(p.text, max_toks, enc)
        for p in m.content if p.type == PartType.tool_result)

def compact_msg(m, user_toks, asst_toks, call_toks, result_toks, enc=None):
    "Render one canonical message."
    if m.role == 'user': return compact_user(m, user_toks, enc)
    if m.role == 'assistant': return compact_asst(m, asst_toks, call_toks, enc)
    if m.role == 'tool': return compact_tool(m, result_toks, enc)

In [ ]:
policy = dict(user_toks=100, asst_toks=100, call_toks=40, result_toks=20)
rendered_turn1 = [compact_msg(m, **policy) for m in turn1]
test_eq(rendered_turn1[0], user)
test_eq(rendered_turn1[1], assistant)
test_eq(rendered_turn1[2], '> files = ["a.py", "b.py"]')
test_eq(rendered_turn1[5], skill)
print('\n'.join(rendered_turn1))

§ List the project files, use the project skill, then run a shell check. §
» I will inspect the project. »
```python
files = ["a.py", "b.py"]
```
> files = ["a.py", "b.py"]
▶ Skill(name='project-files')
> Skill loaded: project-files
§ …compacted… §
! printf "ready\n" ¶ ls -la
> ready


### Conversation rendering

A compact conversation groups messages into user-led turns and separates those turns with `***`. Each message normally uses the bounded policy; the final `last_n` canonical messages use an effectively unlimited policy so the immediate conversational state remains intact.

In [ ]:
#| export
def compact_chat(msgs, user_toks, asst_toks, call_toks, result_toks, enc=None, last_n=0):
    "Render canonical messages as compact user-led turns."
    policy = dict(user_toks=user_toks, asst_toks=asst_toks,
        call_toks=call_toks, result_toks=result_toks)
    full_policy = {k:10**9 for k in policy}
    turns,nfull = [],len(msgs)-last_n
    for i,m in enumerate(msgs):
        if m.role == 'user' or not turns: turns.append([])
        p = full_policy if i >= nfull else policy
        turns[-1].append(compact_msg(m, enc=enc, **p))
    rendered = ('\n'.join(filter(None, turn)) for turn in turns)
    return '\n\n***\n\n'.join(filter(None, rendered))

The sample contains three canonical user messages: the initial request, injected skill text, and the second human request. The skill contents therefore form a short intermediate turn, but are represented only by the compact placeholder.

In [ ]:
compact = compact_chat(conversation, enc=enc, last_n=5, **policy)
test_eq(compact.count('\n\n***\n\n'), 2)
test('§ …skill text compacted: re-invoke before relying on it… §', compact, in_)
test('The fixture is short, readable, and ready for the renderer.', compact, in_)
print(compact)

§ List the project files, use the project skill, then run a shell check. §
» I will inspect the project. »
```python
files = ["a.py", "b.py"]
```
> files = ["a.py", "b.py"]
▶ Skill(name='project-files')
> Skill loaded: project-files

***

§ …compacted… §
! printf "ready\n" ¶ ls -la
> ready

***

§ Keep the agreed names; do not rename the fixture. §
» Understood. I will verify the final state. »
▶ inspect(path='fixture.json', lines=[1, 2])
> first line ¶ second line ¶ final line
» The fixture is short, readable, and ready for the renderer. »


Different message classes have different informational value. User text gets the largest allowance because decisions and corrections cannot safely be reconstructed; assistant prose and tool calls can usually be re-derived; tool results retain a smaller evidential sample. The final five messages are rendered without these limits.

In [ ]:
#| export
compact_policy = dict(user_toks=1000, asst_toks=100, call_toks=50, result_toks=25)

A deliberately small policy makes truncation visible in this short fixture. We set `last_n=0` so every message uses the limits; this is a demonstration policy, not the production default.

In [ ]:
small_policy = dict(user_toks=8, asst_toks=8, call_toks=8, result_toks=6)
small = compact_chat(conversation, last_n=0, **small_policy)
test(' … ', small, in_)
assert len_toks(small) < len_toks(compact)
print(small)

§ List the …  check. §
» I will inspect the project. »
```python
files … "]
```
> files … .py"]
▶ Skill(name='project-files')
> Skill loaded: project-files

***

§ …compacted… §
! printf " …  ls -la
> ready

***

§ Keep the …  fixture. §
» Understood …  state. »
▶ inspect(path …  2])
> first …  final line
» The fixture …  renderer. »


## Compaction documents

The DSL body is wrapped with a short legend and a pointer to the complete transcript. The body supports orientation and reasoning, but it is not authoritative source text: truncation, newline substitution, and later edits mean code and commands must be reopened before reuse.

In [ ]:
#| export
_re_skill = re.compile(r"▶ Skill\(skill='([^']+)'")

def compact_content(chat, path, notes=''):
    "Wrap compact conversation text as a continuation document."
    intro = "This session is being continued from an earlier conversation. `§ … §` encloses user text; `» … »` encloses assistant text; fenced `python` is persistent-kernel execution; `!` marks Bash; `▶` marks another tool call; `>` marks its result; `¶` replaces a newline; `***` separates user-led turns; and ` … ` marks removed middle content. The final five messages are preserved untruncated. This representation is for orientation only: never copy code or commands from it; reopen the original transcript or source file first."
    res = f'{intro}\n\n## Conversation\n\n{chat}'
    if notes: res += f'\n\n## Auto-generated continuation notes\n\n{notes}'
    res += f'\n\nIf further detail is needed, read the complete transcript at: {path}'
    if skills := L(_re_skill.findall(chat)).unique():
        res += '\n\nSkill contents were compacted out above. Before relying on any skill guidance, re-invoke it: ' + ', '.join(f'Skill({s})' for s in skills) + '.'
    return res

`compact_content` adds the legend, a `## Conversation` section, an optional notes section, and the path of the authoritative transcript.

In [ ]:
content = compact_content(small, '/tmp/session.jsonl')
test('## Conversation', content, in_)
test('/tmp/session.jsonl', content, in_)
print(content.split('\n\n## Conversation', 1)[0])

This session is being continued from an earlier conversation. `§ … §` encloses user text; `» … »` encloses assistant text; fenced `python` is persistent-kernel execution; `!` marks Bash; `▶` marks another tool call; `>` marks its result; `¶` replaces a newline; `***` separates user-led turns; and ` … ` marks removed middle content. The final five messages are preserved untruncated. This representation is for orientation only: never copy code or commands from it; reopen the original transcript or source file first.


In [ ]:
#| export
def compact_body(content):
    "Extract the conversation body from a compaction document."
    body = content.split('\n\n## Conversation\n\n', 1)[1]
    body = body.split('\n\n## Auto-generated continuation notes\n\n', 1)[0]
    return body.rsplit('\n\nIf further detail is needed, read the complete transcript at:', 1)[0]

`compact_body` removes the document wrapper and optional continuation notes, recovering only the conversation DSL. This lets repeated compaction retain an earlier immutable segment without recursively truncating it.

In [ ]:
test_eq(compact_body(content), small)
with_notes = compact_content(small, '/tmp/session.jsonl', 'A useful note.')
test_eq(compact_body(with_notes), small)

Skill loads are the one thing the DSL compacts away entirely, so the document ends by instructing their re-invocation. The list derives from the `▶ Skill(...)` calls preserved in the body - including calls in earlier immutable segments, so repeated compaction keeps the union - and `compact_body` discards the instruction along with the rest of the wrapper.

In [ ]:
tailed = compact_content(compact, '/tmp/session.jsonl')
test('re-invoke it: Skill(project-files)', tailed, in_)
test_eq(compact_body(tailed), compact)
assert 'Skill(' not in compact_content('§ hi §', '/tmp/session.jsonl')

In [ ]:
#| export
def join_compacts(*parts):
    "Join non-empty compact conversation segments."
    return '\n\n***\n\n'.join(part for part in parts if part)

`join_compacts` concatenates non-empty conversation segments with the same `***` separator used between user-led turns. Empty segments are ignored, which handles sessions with no earlier synthetic compaction.

In [ ]:
joined = join_compacts('first', '', 'second')
test_eq(joined, 'first\n\n***\n\nsecond')
print(joined)

first

***

second


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()